In [43]:
import pandas as pd
import numpy as np
from pathlib import Path
import yfinance as yf
import simfin as sf

ROOT = Path.cwd().resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"

for p in [DATA_RAW, DATA_PROCESSED]:
    p.mkdir(parents=True, exist_ok=True)

In [44]:
sp500_url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
sp500_df = pd.read_csv(sp500_url)
universe_500 = (
    sp500_df["Symbol"]
    .astype(str)
    .str.replace(".", "-", regex=False)
    .tolist()[:500]
)

pd.DataFrame({"ticker": universe_500}).to_csv(DATA_RAW / "universe_500.csv", index=False)
print("Universe 500:", len(universe_500))
print(universe_500[:10])

Universe 500: 500
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


In [45]:
sf.set_api_key("e9de6406-d944-4792-98bd-bb5059dc9749")
sf.set_data_dir("simfin/data")

income = sf.load_income(variant="quarterly", market="us").reset_index()
balance = sf.load_balance(variant="quarterly", market="us").reset_index()

income = income.rename(columns={"Ticker": "ticker", "Report Date": "report_date"})
balance = balance.rename(columns={"Ticker": "ticker", "Report Date": "report_date"})

income_500 = income[income["ticker"].isin(universe_500)].copy()
balance_500 = balance[balance["ticker"].isin(universe_500)].copy()

print("Income 500 shape:", income_500.shape)
print("Balance 500 shape:", balance_500.shape)
print("Income tickers:", income_500["ticker"].nunique())
print("Balance tickers:", balance_500["ticker"].nunique())

Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Income 500 shape: (6193, 28)
Balance 500 shape: (6195, 30)
Income tickers: 358
Balance tickers: 358


In [46]:
fundamentals_500 = income_500[
    ["ticker", "report_date", "Revenue", "Cost of Revenue", "Net Income", "Shares (Diluted)"]
].copy()

fundamentals_500 = fundamentals_500.rename(columns={
    "Revenue": "revenue",
    "Cost of Revenue": "cogs",
    "Net Income": "net_income",
    "Shares (Diluted)": "shares_diluted"
})

balance_small_500 = balance_500[
    ["ticker", "report_date", "Total Equity", "Total Assets"]
].copy()

balance_small_500 = balance_small_500.rename(columns={
    "Total Equity": "book_value",
    "Total Assets": "total_assets"
})

fundamentals_500 = fundamentals_500.merge(
    balance_small_500,
    on=["ticker", "report_date"],
    how="left"
)

fundamentals_500.to_csv(DATA_RAW / "fundamentals_500_raw.csv", index=False)

coverage_500 = (
    fundamentals_500.groupby("ticker")
    .agg(
        n_quarters=("report_date", "nunique"),
        n_rows=("report_date", "size"),
        has_book_value=("book_value", lambda s: s.notna().sum()),
        has_net_income=("net_income", lambda s: s.notna().sum())
    )
    .reset_index()
)

coverage_500.to_csv(DATA_PROCESSED / "coverage_500.csv", index=False)

universe_358 = coverage_500.loc[
    (coverage_500["n_quarters"] >= 4) &
    (coverage_500["has_book_value"] > 0) &
    (coverage_500["has_net_income"] > 0),
    "ticker"
].sort_values().tolist()

pd.DataFrame({"ticker": universe_358}).to_csv(DATA_RAW / "universe_358.csv", index=False)

fundamentals_358 = fundamentals_500[fundamentals_500["ticker"].isin(universe_358)].copy()
fundamentals_358.to_csv(DATA_RAW / "fundamentals_358_raw.csv", index=False)

print("Universe 358:", len(universe_358))
print("Fundamentals 500 rows:", len(fundamentals_500))
print("Fundamentals 358 rows:", len(fundamentals_358))

Universe 358: 327
Fundamentals 500 rows: 6193
Fundamentals 358 rows: 6146


In [ ]:
prices = yf.download(
    universe_500,
    start="2014-01-01",
    end="2026-06-01",
    auto_adjust=False,
    progress=False,
    group_by="ticker",
    threads=True
)

print(prices.columns)

if isinstance(prices.columns, pd.MultiIndex):
    #multiindex format w/ level 0 = ticker and level 1 = price field
    if "Adj Close" in prices.columns.get_level_values(1):
        adjclose = prices.xs("Adj Close", axis=1, level=1).copy()
    elif "Close" in prices.columns.get_level_values(1):
        adjclose = prices.xs("Close", axis=1, level=1).copy()
    else:
        raise KeyError(f"Available price fields: {prices.columns.get_level_values(1).unique().tolist()}")
else:
    #single level columns
    if "Adj Close" in prices.columns:
        adjclose = prices["Adj Close"].copy()
    elif "Close" in prices.columns:
        adjclose = prices["Close"].copy()
    else:
        raise KeyError(f"Available columns: {prices.columns.tolist()}")

adjclose.to_csv(DATA_RAW / "prices_adjclose_500.csv")

monthly_prices_500 = adjclose.resample("ME").last()
monthly_returns_500 = monthly_prices_500.pct_change().dropna(how="all")

monthly_prices_500.to_csv(DATA_PROCESSED / "monthly_prices_500.csv")
monthly_returns_500.to_csv(DATA_PROCESSED / "monthly_returns_500.csv")

print("Adj close shape:", adjclose.shape)
print("Monthly prices shape:", monthly_prices_500.shape)
print("Monthly returns shape:", monthly_returns_500.shape)

MultiIndex([('FDXF',      'Open'),
            ('FDXF',      'High'),
            ('FDXF',       'Low'),
            ('FDXF',     'Close'),
            ('FDXF', 'Adj Close'),
            ('FDXF',    'Volume'),
            ('CIEN',      'Open'),
            ('CIEN',      'High'),
            ('CIEN',       'Low'),
            ('CIEN',     'Close'),
            ...
            ('TTWO',       'Low'),
            ('TTWO',     'Close'),
            ('TTWO', 'Adj Close'),
            ('TTWO',    'Volume'),
            (  'CB',      'Open'),
            (  'CB',      'High'),
            (  'CB',       'Low'),
            (  'CB',     'Close'),
            (  'CB', 'Adj Close'),
            (  'CB',    'Volume')],
           names=['Ticker', 'Price'], length=3000)
Adj close shape: (3120, 500)
Monthly prices shape: (149, 500)
Monthly returns shape: (148, 500)


In [48]:
prices = yf.download(
    universe_500,
    start="2014-01-01",
    end="2026-06-01",
    auto_adjust=False,
    progress=False,
    group_by="ticker",
    threads=True
)

if isinstance(prices.columns, pd.MultiIndex):
    if "Adj Close" in prices.columns.get_level_values(1):
        adjclose = prices.xs("Adj Close", axis=1, level=1).copy()
    elif "Close" in prices.columns.get_level_values(1):
        adjclose = prices.xs("Close", axis=1, level=1).copy()
    else:
        raise KeyError(f"Available price fields: {prices.columns.get_level_values(1).unique().tolist()}")
else:
    if "Adj Close" in prices.columns:
        adjclose = prices["Adj Close"].copy()
    elif "Close" in prices.columns:
        adjclose = prices["Close"].copy()
    else:
        raise KeyError(f"Available columns: {prices.columns.tolist()}")

adjclose = adjclose.sort_index()
adjclose = adjclose.dropna(axis=1, how="all")

adjclose.to_csv(DATA_RAW / "prices_adjclose_500.csv")

monthly_prices_500 = adjclose.resample("ME").last()
monthly_returns_500 = monthly_prices_500.pct_change().dropna(how="all")

monthly_prices_500.to_csv(DATA_PROCESSED / "monthly_prices_500.csv")
monthly_returns_500.to_csv(DATA_PROCESSED / "monthly_returns_500.csv")

print("Adj close shape:", adjclose.shape)
print("Monthly prices shape:", monthly_prices_500.shape)
print("Monthly returns shape:", monthly_returns_500.shape)
print("Price tickers:", adjclose.columns.nunique())

Adj close shape: (3120, 500)
Monthly prices shape: (149, 500)
Monthly returns shape: (148, 500)
Price tickers: 500


In [49]:
return_panel_500 = monthly_returns_500.stack().reset_index()
return_panel_500.columns = ["date", "ticker", "return"]
return_panel_500 = return_panel_500.sort_values(["ticker", "date"]).reset_index(drop=True)
return_panel_500.to_csv(DATA_PROCESSED / "return_panel_500.csv", index=False)

print("Return panel 500 shape:", return_panel_500.shape)
print(return_panel_500.head())

Return panel 500 shape: (74000, 3)
        date ticker    return
0 2014-02-28      A -0.020980
1 2014-03-31      A -0.017741
2 2014-04-30      A -0.031280
3 2014-05-31      A  0.053664
4 2014-06-30      A  0.011090


In [50]:
momentum_12m_500 = monthly_prices_500.pct_change(12).shift(1)
momentum_12m_500.to_csv(DATA_PROCESSED / "momentum_12m_500.csv")

momentum_panel_500 = momentum_12m_500.stack().reset_index()
momentum_panel_500.columns = ["date", "ticker", "momentum_12m"]
momentum_panel_500 = momentum_panel_500.dropna().sort_values(["ticker", "date"]).reset_index(drop=True)
momentum_panel_500.to_csv(DATA_PROCESSED / "momentum_panel_500.csv", index=False)

print("Momentum 12m shape:", momentum_12m_500.shape)
print("Momentum panel shape:", momentum_panel_500.shape)
print(momentum_panel_500.head())

Momentum 12m shape: (149, 500)
Momentum panel shape: (65148, 3)
        date ticker  momentum_12m
0 2015-02-28      A     -0.083319
1 2015-03-31      A      0.046393
2 2015-04-30      A      0.051219
3 2015-05-31      A      0.080462
4 2015-06-30      A      0.020971


In [51]:
fundamentals_358 = fundamentals_500[fundamentals_500["ticker"].isin(universe_358)].copy()
fundamentals_358.to_csv(DATA_RAW / "fundamentals_358_raw.csv", index=False)

coverage_358 = (
    fundamentals_358.groupby("ticker")
    .agg(
        n_quarters=("report_date", "nunique"),
        has_book_value=("book_value", lambda s: s.notna().sum()),
        has_net_income=("net_income", lambda s: s.notna().sum())
    )
    .reset_index()
)

coverage_358.to_csv(DATA_PROCESSED / "coverage_358.csv", index=False)

print("Coverage 358 shape:", coverage_358.shape)
print("Coverage 358 tickers:", coverage_358["ticker"].nunique())

Coverage 358 shape: (327, 4)
Coverage 358 tickers: 327


In [52]:
manifest = pd.DataFrame([
    {"file": "data/raw/universe_500.csv", "description": "All S&P 500 tickers"},
    {"file": "data/raw/universe_358.csv", "description": "Core fundamental universe"},
    {"file": "data/raw/fundamentals_500_raw.csv", "description": "Raw fundamentals for 500"},
    {"file": "data/raw/fundamentals_358_raw.csv", "description": "Raw fundamentals for core 358"},
    {"file": "data/raw/prices_adjclose_500.csv", "description": "Yahoo adjusted close prices"},
    {"file": "data/processed/monthly_prices_500.csv", "description": "Monthly prices"},
    {"file": "data/processed/monthly_returns_500.csv", "description": "Monthly returns"},
    {"file": "data/processed/return_panel_500.csv", "description": "Long-format monthly returns"},
    {"file": "data/processed/momentum_12m_500.csv", "description": "12-month momentum matrix"},
    {"file": "data/processed/momentum_panel_500.csv", "description": "Long-format momentum"},
    {"file": "data/processed/coverage_500.csv", "description": "Coverage stats for 500"},
    {"file": "data/processed/coverage_358.csv", "description": "Coverage stats for 358"},
])

manifest.to_csv(DATA_PROCESSED / "pipeline_manifest.csv", index=False)
print(manifest)

                                      file                    description
0                data/raw/universe_500.csv            All S&P 500 tickers
1                data/raw/universe_358.csv      Core fundamental universe
2        data/raw/fundamentals_500_raw.csv       Raw fundamentals for 500
3        data/raw/fundamentals_358_raw.csv  Raw fundamentals for core 358
4         data/raw/prices_adjclose_500.csv    Yahoo adjusted close prices
5    data/processed/monthly_prices_500.csv                 Monthly prices
6   data/processed/monthly_returns_500.csv                Monthly returns
7      data/processed/return_panel_500.csv    Long-format monthly returns
8      data/processed/momentum_12m_500.csv       12-month momentum matrix
9    data/processed/momentum_panel_500.csv           Long-format momentum
10         data/processed/coverage_500.csv         Coverage stats for 500
11         data/processed/coverage_358.csv         Coverage stats for 358
